# Visualise differences between Jowo and FOIS

In [2]:
import os
import pandas as pd

In [3]:
df_path = os.path.join("..","..","..", "data", "raw", "conferences","ontology","jowo_fois_with_abstracts.csv")
df = pd.read_csv(df_path)

## 1. Check availability

In [4]:
missing = df["keywords_pdf"].isna() | (df["keywords_pdf"].str.strip() == "")
print(f"{missing.sum()} / {len(df)} ({missing.mean()*100:.1f}%)")

36 / 637 (5.7%)


In [5]:
df["kw_list"] = df["keywords_pdf"].str.split(",")
df_exploded = df.explode("kw_list")
df_exploded["kw_clean"] = df_exploded["kw_list"].str.strip().str.lower()

freq = (
    df_exploded.groupby(["year", "venue", "kw_clean"])
    .size()
    .reset_index(name="count")
)

In [6]:
len(freq)

2594

## Cluster 

Identify Clusters

In [8]:
from tqdm import tqdm
import ollama
import numpy as np

keywords = freq["kw_clean"].unique().tolist()

embeddings = []
batch_size = 64

for i in tqdm(range(0, len(keywords), batch_size), desc="Embeddings"):
    batch = keywords[i:i+batch_size]
    resp = ollama.embed(model="embeddinggemma", input=batch)
    embeddings.extend(resp["embeddings"])

X = np.array(embeddings)

Embeddings: 100%|██████████| 31/31 [00:26<00:00,  1.17it/s]


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

X_norm = normalize(X)
X_reduced = PCA(n_components=30, random_state=42).fit_transform(X_norm)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

scores = []
K_range = range(5, 25)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_reduced)
    score = silhouette_score(X_reduced, lbl, sample_size=500)
    scores.append((k, score))
    print(f"k={k:2d} → Silhouette: {score:.4f}")

best_k = max(scores, key=lambda x: x[1])[0]
print(f"\nBestes k: {best_k}")

k= 5 → Silhouette: 0.1053
k= 6 → Silhouette: 0.0935
k= 7 → Silhouette: 0.0815
k= 8 → Silhouette: 0.0894
k= 9 → Silhouette: 0.0760
k=10 → Silhouette: 0.0783
k=11 → Silhouette: 0.0717
k=12 → Silhouette: 0.0654
k=13 → Silhouette: 0.0891
k=14 → Silhouette: 0.0858
k=15 → Silhouette: 0.0691
k=16 → Silhouette: 0.0791
k=17 → Silhouette: 0.0771
k=18 → Silhouette: 0.0766
k=19 → Silhouette: 0.0758
k=20 → Silhouette: 0.0829
k=21 → Silhouette: 0.0759
k=22 → Silhouette: 0.0922
k=23 → Silhouette: 0.0762
k=24 → Silhouette: 0.0735

Bestes k: 5


In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km.fit_predict(X_reduced)

kw_cluster_map = dict(zip(keywords, labels))
freq["cluster_id"] = freq["kw_clean"].map(kw_cluster_map)

for cid in range(best_k):
    top = (freq[freq["cluster_id"] == cid]
           .groupby("kw_clean")["count"].sum()
           .sort_values(ascending=False)
           .head(8).index.tolist())
    print(f"\nCluster {cid}: {top}")


Cluster 0: ['knowledge graph', 'semantic web', 'knowledge representation', 'knowledge graphs', 'image schemas', 'digital humanities', 'large language models', 'machine learning']

Cluster 1: ['ontology', 'basic formal ontology', 'ontologies', 'ontology engineering', 'applied ontology', 'foundational ontology', 'basic formal ontology (bfo)', 'formal ontology']

Cluster 2: ['mereology', 'realizable entity', 'semantics', 'mereotopology', 'representation', 'concepts', 'cognition', 'conceptual spaces']

Cluster 3: ['disposition', 'owl', '', 'role', 'ufo', 'function', 'bfo', 'ontouml']

Cluster 4: ['conceptual modeling', 'formal modeling', 'modeling', 'models for manufacturing', 'energy systems modelling', 'enterprise modeling', 'model', 'modelling']


In [ ]:
cluster_meta = {
    0: "Knowledge Graphs & AI",
    1: "Ontology Engineering",
    2: "Mereology & Cognition",
    3: "Formal Ontology Concepts",
    4: "Conceptual Modeling"
}

km = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = km.fit_predict(X_reduced)

kw_cluster_map = dict(zip(keywords, labels))
freq["cluster_id"] = freq["kw_clean"].map(kw_cluster_map)
freq["cluster"] = freq["cluster_id"].map(cluster_meta)

# Leere Keywords raus
freq = freq[freq["kw_clean"].str.strip() != ""]

print(freq["cluster"].value_counts())


cluster
Formal Ontology Concepts    934
Knowledge Graphs & AI       664
Mereology & Cognition       513
Ontology Engineering        402
Conceptual Modeling          74
Name: count, dtype: int64


In [14]:
freq_path = os.path.join("..","..","..", "data", "raw", "conferences","ontology","jowo_fois_cluster_keywords.csv")
freq.to_csv(freq_path)